# Assay ↔ DILI overlap: the counts, computed live

**Why this notebook exists.** Two different overlap numbers circulate for this project and
they look contradictory:

| Claim | Number |
|---|---|
| "Held-223 barely overlaps MMP" | 27.9% (62/222) |
| "MMP mostly overlaps DILI" | 76.0% (4513/5937) |

Both are correct. They are the **same intersections** read in **opposite directions**, over
**different DILI sets**. This notebook computes every cell so you can see it.

Run from the `DILI_assay_link/` folder.

## 1. Load the compound sets (by InChIKey, never SMILES)

In [10]:
import pandas as pd, sys
sys.path.insert(0, '.')

# the exact InChIKeys the pipeline trained on
A = {a: set(open(f'data_{a}/ikeys_all.txt').read().split()) for a in ('mmp', 'caspase')}

d = pd.read_csv('data_dili_features/dili_labels.csv')
S = {g: set(x.InChIKey) for g, x in d.groupby('dili_set')}
S['Gold-1111 (=NCV+Held)'] = S['NCV-888'] | S['Held-223']
S['FULL DILI universe']    = set(d.InChIKey)

for k, v in A.items(): print(f'assay  {k:9s} {len(v):6d}')
print()
for k in ['Held-223','NCV-888','Gold-1111 (=NCV+Held)','Alloriginal','FULL DILI universe']:
    print(f'DILI   {k:24s} {len(S[k]):6d}')

assay  mmp         5937
assay  caspase     7282

DILI   Held-223                    222
DILI   NCV-888                     874
DILI   Gold-1111 (=NCV+Held)      1096
DILI   Alloriginal               12145
DILI   FULL DILI universe        13241


### Optional: add Caspase CHO-K1

CHO-K1 was dropped from the linking experiment (different cell line), so this folder has no
cached InChIKeys for it. This cell derives them from the cleaned assay CSV — takes ~30 s.
Skip it if you only care about the two HepG2 assays.

In [11]:
ADD_CHOK1 = True

if ADD_CHOK1:
    from features import canonicalize, smiles_to_inchikey
    c = pd.read_csv('../summary_1347037/cleaning/AID_1347037_cleaned.csv')
    smi_col = [x for x in c.columns if 'SMILES' in x.upper()][0]
    ks = set()
    for s in c[smi_col].dropna().astype(str):
        try:
            k = smiles_to_inchikey(canonicalize(s))
            if k: ks.add(k)
        except Exception:
            pass
    A['chok1'] = ks
    print(f'chok1 {len(ks)} unique InChIKeys from {len(c)} rows')

chok1 7670 unique InChIKeys from 7671 rows


## 2. Direction 1 — "how much of the DILI set is in the assay?"

This is the table that gives the *small* percentages. Denominator = the DILI set.

In [12]:
NAME  = {'mmp':'MMP', 'caspase':'Caspase HepG2', 'chok1':'Caspase CHO-K1'}
ORDER = ['Held-223','NCV-888','Gold-1111 (=NCV+Held)','Alloriginal','FULL DILI universe']

rows = []
for g in ORDER:
    r = {'DILI set': g, 'n': len(S[g])}
    for a in A:
        k = len(S[g] & A[a])
        r[NAME[a]] = f'{100*k/len(S[g]):.1f}%  ({k})'
    rows.append(r)
pd.DataFrame(rows).set_index('DILI set')

,n,MMP,Caspase HepG2,Caspase CHO-K1
DILI set,,,,
Held-223,222,27.9% (62),35.6% (79),38.3% (85)
NCV-888,874,19.2% (168),26.1% (228),28.0% (245)
Gold-1111 (=NCV+Held),1096,21.0% (230),28.0% (307),30.1% (330)
Alloriginal,12145,35.3% (4283),38.9% (4730),40.9% (4970)
FULL DILI universe,13241,34.1% (4513),38.0% (5037),40.0% (5300)


## 3. Direction 2 — "how much of the assay is in the DILI set?"

Same intersections. Denominator = the **assay**. This is where 76% / 69% comes from.

In [13]:
rows = []
for a in A:
    r = {'Assay': NAME[a], 'n': len(A[a])}
    for g in ORDER:
        k = len(S[g] & A[a])
        r[g] = f'{100*k/len(A[a]):.1f}%  ({k})'
    rows.append(r)
pd.DataFrame(rows).set_index('Assay')

,n,Held-223,NCV-888,Gold-1111 (=NCV+Held),Alloriginal,FULL DILI universe
Assay,,,,,,
MMP,5937,1.0% (62),2.8% (168),3.9% (230),72.1% (4283),76.0% (4513)
Caspase HepG2,7282,1.1% (79),3.1% (228),4.2% (307),65.0% (4730),69.2% (5037)
Caspase CHO-K1,7670,1.1% (85),3.2% (245),4.3% (330),64.8% (4970),69.1% (5300)


## 4. Where the 76% actually comes from

Note in the table above that for MMP:

- Gold-1111 (the 1,096 labelled DILI compounds) accounts for only **3.9%**
- **Alloriginal** (the 12,145-compound clinical pool) accounts for **72.1%**

**This is the whole point.** The DILI models do not train on Gold-1111 — they train on the
*clinical pool*. So the set a compound must be kept out of is the FULL DILI universe
(13,241), not the gold sets. Your pasted table looked only at the gold sets, which is why
it showed 20–38% and looked harmless.

In [14]:
for a in A:
    tot  = len(S['FULL DILI universe'] & A[a])
    gold = len(S['Gold-1111 (=NCV+Held)'] & A[a])
    allo = len(S['Alloriginal'] & A[a])
    print(f'{NAME[a]:15s} overlap {tot:5d} / {len(A[a]):5d} = {100*tot/len(A[a]):5.1f}%'
          f'   of which Alloriginal {allo:5d} ({100*allo/len(A[a]):4.1f}%)'
          f'   Gold-1111 {gold:4d} ({100*gold/len(A[a]):4.1f}%)')

MMP             overlap  4513 /  5937 =  76.0%   of which Alloriginal  4283 (72.1%)   Gold-1111  230 ( 3.9%)
Caspase HepG2   overlap  5037 /  7282 =  69.2%   of which Alloriginal  4730 (65.0%)   Gold-1111  307 ( 4.2%)
Caspase CHO-K1  overlap  5300 /  7670 =  69.1%   of which Alloriginal  4970 (64.8%)   Gold-1111  330 ( 4.3%)


## 5. Why exclusion was impossible

If we had excluded every DILI compound from assay training, this is what would be left.

In [15]:
import numpy as np
for a in ('mmp','caspase'):
    ik = open(f'data_{a}/ikeys_all.txt').read().split()
    y  = np.load(f'data_{a}/y_all.npy')
    keep = np.array([k not in S['FULL DILI universe'] for k in ik])
    print(f'{NAME[a]:15s} full: {len(y):5d} compounds, {int(y.sum()):4d} actives '
          f'({100*y.mean():4.1f}%)')
    print(f'{"":15s} after excluding all DILI: {int(keep.sum()):5d} compounds, '
          f'{int(y[keep].sum()):4d} actives ({100*y[keep].mean():4.1f}%)')

MMP             full:  5937 compounds,  857 actives (14.4%)
                after excluding all DILI:  1424 compounds,  205 actives (14.4%)
Caspase HepG2   full:  7282 compounds,  281 actives ( 3.9%)
                after excluding all DILI:  2245 compounds,  140 actives ( 6.2%)


## 6. The OOB fix, verified

Instead of excluding, each compound's in vitro value is averaged over **only the repeats
where it was held out**. `oob_mask.npy` is `[21, n_compounds]`, True = held out that repeat.

The checks below are the ones `audit_leakage.py` runs on the cluster.

In [16]:
for a in ('mmp','caspase'):
    m  = np.load(f'data_{a}/oob_mask.npy')
    ik = np.array(open(f'data_{a}/ikeys_all.txt').read().split())
    cov = m.sum(axis=0)
    ov  = np.isin(ik, list(S['FULL DILI universe']))
    print(f'{NAME[a]:15s} mask {m.shape}  every compound held out >=1x: {bool((cov>0).all())}')
    print(f'{"":15s} DILI-overlapping compounds: {int(ov.sum())}, '
          f'mean OOB repeats each {cov[ov].mean():.2f} (min {cov[ov].min()})')

MMP             mask (21, 5937)  every compound held out >=1x: True
                DILI-overlapping compounds: 4513, mean OOB repeats each 4.18 (min 1)
Caspase HepG2   mask (21, 7282)  every compound held out >=1x: True
                DILI-overlapping compounds: 5037, mean OOB repeats each 4.22 (min 1)


### Cross-check against the shipped feature file

`invitro_features.csv` stores a provenance count per compound. For a compound **in** the
assay it must be < 21 (only its held-out repeats counted); for a compound **absent** from
the assay it must be exactly 21 (all repeats legal).

In [17]:
f = pd.read_csv('results/invitro_features.csv')
for a in ('mmp','caspase'):
    inassay = f.InChIKey.isin(A[a])
    print(f'{NAME[a]:15s} in assay  (n={inassay.sum():5d}): '
          f'{a}_n_oob_repeats  max={f.loc[inassay, f"{a}_n_oob_repeats"].max():2d}  '
          f'mean={f.loc[inassay, f"{a}_n_oob_repeats"].mean():5.2f}')
    print(f'{"":15s} not in it (n={(~inassay).sum():5d}): '
          f'max={f.loc[~inassay, f"{a}_n_oob_repeats"].max():2d}  '
          f'mean={f.loc[~inassay, f"{a}_n_oob_repeats"].mean():5.2f}')

MMP             in assay  (n= 4513): mmp_n_oob_repeats  max=12  mean= 4.18
                not in it (n= 8728): max=21  mean=21.00
Caspase HepG2   in assay  (n= 5037): caspase_n_oob_repeats  max=11  mean= 4.22
                not in it (n= 8204): max=21  mean=21.00


## 7. Caveat on what actually ran

`invitro_summary.csv` records how many assay model families contributed to each feature.

In [18]:
print(pd.read_csv('results/invitro_summary.csv').to_string(index=False))
print()
import os
for a in ('mmp','caspase'):
    fams = sorted({x.split('_r')[0] for x in os.listdir(f'results/{a}/probs')
                   if x.endswith('_dili.npy')})
    print(f'{a:9s} families with DILI predictions: {fams}')

  assay  families  repeats  overlap  featured  mean_prob  mean_repeats
    mmp         2       21     4513     13241     0.2935         15.27
caspase         2       21     5037     13241     0.1529         14.62

mmp       families with DILI predictions: ['RF', 'XGB']
caspase   families with DILI predictions: ['RF', 'XGB']


`families = 2` means `mmp_prob` and `caspase_prob` are a **RF + XGB average**, not the
10-model consensus the docstring in `build_invitro_features.py` describes. The two tree
models do reach 0.886 / 0.894 (MMP) and 0.865 / 0.844 (caspase) ROC-AUC, so the features
carry real signal — but the remaining 8 families never produced DILI predictions.

To close that: re-run `submit_assays.sh` for the missing families, then
`build_invitro_features.py` → `train_dili_link.py` → `aggregate_link_results.py`.

---
# 8. Three-way overlap: do the MEASURED assay labels associate with DILI?

Everything above is about *leakage control*. This section asks the biological question
directly, with **no machine learning at all** — just the measured 0/1 assay labels
cross-tabulated against the DILI outcome.

In [19]:
import numpy as np
from scipy import stats

lab = {a: pd.Series(np.load(f'data_{a}/y_all.npy'),
                    index=np.array(open(f'data_{a}/ikeys_all.txt').read().split()))
       for a in ('mmp', 'caspase')}
dl = pd.read_csv('data_dili_features/dili_labels.csv').set_index('InChIKey')

t = pd.DataFrame({'DILI': dl.TOXICITY, 'set': dl.dili_set})
t['mmp'] = lab['mmp'].reindex(t.index)
t['cas'] = lab['caspase'].reindex(t.index)

print(f'DILI compounds                 {len(t)}')
print(f'  with a measured MMP label    {t.mmp.notna().sum()}')
print(f'  with a measured casp label   {t.cas.notna().sum()}')
print(f'  with BOTH (the 3-way set)    {len(t.dropna(subset=["mmp","cas"]))}')

DILI compounds                 13241
  with a measured MMP label    4513
  with a measured casp label   5037
  with BOTH (the 3-way set)    3853


### 8a. Each assay on its own

In [20]:
for col, nm in (('mmp', 'MMP'), ('cas', 'Caspase')):
    g  = t.dropna(subset=[col])
    ct = pd.crosstab(g[col].astype(int), g.DILI)
    ct.index = [f'{nm} inactive', f'{nm} ACTIVE']; ct.columns = ['DILI-', 'DILI+']
    ct['DILI+ %'] = (100 * ct['DILI+'] / ct.sum(axis=1)).round(1)
    odds, p = stats.fisher_exact(ct.iloc[:, :2].values)
    print(f'--- {nm} ---'); print(ct.to_string())
    print(f'  Fisher OR {odds:.2f}   p = {p:.3g}\n')

--- MMP ---
              DILI-  DILI+  DILI+ %
MMP inactive   3069    792     20.5
MMP ACTIVE      166    486     74.5
  Fisher OR 11.34   p = 1.49e-158

--- Caspase ---
                  DILI-  DILI+  DILI+ %
Caspase inactive   3443   1453     29.7
Caspase ACTIVE       62     79     56.0
  Fisher OR 3.02   p = 1.74e-10



### 8b. The four-cell breakdown — does being active in BOTH matter?

In [21]:
b = t.dropna(subset=['mmp', 'cas']).copy()
b['cell'] = np.where(b.mmp.astype(bool) & b.cas.astype(bool), 'MMP+ & Casp+',
            np.where(b.mmp.astype(bool), 'MMP+ only',
            np.where(b.cas.astype(bool), 'Casp+ only', 'neither')))
r = b.groupby('cell').DILI.agg(n='size', pos='sum')
r['DILI+ %'] = (100 * r.pos / r.n).round(1)
display(r.sort_values('DILI+ %', ascending=False))

# is caspase informative CONDITIONAL on MMP?
for m in (0, 1):
    g  = b[b.mmp == m]
    ct = pd.crosstab(g.cas.astype(int), g.DILI)
    if ct.shape == (2, 2):
        odds, p = stats.fisher_exact(ct.values)
        print(f'caspase effect given MMP={m}:  OR {odds:.2f}   p = {p:.3g}')

ct = pd.crosstab(b.mmp.astype(int), b.cas.astype(int))
odds, p = stats.fisher_exact(ct.values)
print(f'\ndo the two assays co-occur?  OR {odds:.2f}  p = {p:.3g}  '
      f'(high OR = caspase is largely redundant with MMP)')

,n,pos,DILI+ %
cell,,,
MMP+ only,420,320,76.2
MMP+ & Casp+,57,40,70.2
Casp+ only,14,4,28.6
neither,3362,692,20.6


caspase effect given MMP=0:  OR 1.54   p = 0.505
caspase effect given MMP=1:  OR 0.74   p = 0.327

do the two assays co-occur?  OR 32.59  p = 4.29e-40  (high OR = caspase is largely redundant with MMP)


### 8c. The decisive check — stratify by DILI subset

The three DILI subsets have very different baseline DILI+ rates (33% / 63% / 70%), so a
pooled association can be driven entirely by one subset. **Held-223 is the set the models
are actually scored on.**

In [22]:
for col, nm in (('mmp', 'MMP'), ('cas', 'Caspase')):
    print(f'=== {nm} ===')
    for s in ['Alloriginal', 'NCV-888', 'Held-223']:
        g  = t[t.set == s].dropna(subset=[col])
        ct = pd.crosstab(g[col].astype(int), g.DILI)
        if ct.shape != (2, 2):
            print(f'  {s:12s} too few actives for a 2x2'); continue
        inact = 100 * ct.loc[0, 1] / ct.loc[0].sum()
        act   = 100 * ct.loc[1, 1] / ct.loc[1].sum()
        odds, p = stats.fisher_exact(ct.values)
        print(f'  {s:12s} n={len(g):5d}  inactive {inact:5.1f}% DILI+  '
              f'ACTIVE {act:5.1f}% DILI+ (n={ct.loc[1].sum():4d})  OR {odds:6.2f}  p={p:.2e}')
    print()

=== MMP ===
  Alloriginal  n= 4283  inactive  18.0% DILI+  ACTIVE  74.5% DILI+ (n= 627)  OR  13.27  p=7.70e-170
  NCV-888      n=  168  inactive  61.6% DILI+  ACTIVE  76.5% DILI+ (n=  17)  OR   2.03  p=2.94e-01
  Held-223     n=   62  inactive  74.1% DILI+  ACTIVE  75.0% DILI+ (n=   8)  OR   1.05  p=1.00e+00

=== Caspase ===
  Alloriginal  n= 4730  inactive  27.1% DILI+  ACTIVE  53.3% DILI+ (n= 120)  OR   3.07  p=2.73e-09
  NCV-888      n=  228  inactive  66.7% DILI+  ACTIVE  73.3% DILI+ (n=  15)  OR   1.38  p=7.78e-01
  Held-223     n=   79  inactive  82.2% DILI+  ACTIVE  66.7% DILI+ (n=   6)  OR   0.43  p=3.19e-01



### What this means

The MMP↔DILI association is enormous in **Alloriginal** (OR 13.3) and **absent in
Held-223** (75.0% vs 74.1%, OR 1.05, p = 1.00).

That is the mechanistic explanation for the null linking result in
`results/link/linking_comparison.csv`: the DILI models are *evaluated on Held-223*, and on
that set MMP activity carries no information about the outcome. No feature derived from the
MMP assay can improve test performance, however good the assay model is.

Two honest caveats on that conclusion:
- only **8** of the 62 MMP-overlapping Held-223 compounds are MMP-active — underpowered
- Held-223 is already **69.8% DILI+**, so there is a ceiling effect limiting visible enrichment